In [ ]:
import sys; sys.path.insert(0, 'calibration')  # utils_calibrate* live in calibration/

In [ ]:
# %matplotlib inline
import numpy as np
import electric as electric
import utils_calibrate2
import importlib

importlib.reload(electric)
importlib.reload(utils_calibrate2)
from utils_calibrate2 import (
    SimpleEFishAgent,
    plot_image_fish2,
    plot_line_fish2,
    get_vmin_vmax,
    run_self_image_experiment,
    run_direct_eod_sensing_experiment_no_food,
    run_direct_eod_sensing_experiment_no_food_fish2_grid,
    run_direct_eod_sensing_experiment_with_food,
    run_self_image_active_sense_other_fish_experiment,
    run_self_image_experiment_with_without_food
)
import matplotlib.pyplot as plt
import cfg
from cfg import AGENT_PARAMS, ENV_PARAMS, ELECTRIC_CONSTANTS, cm_to_m, m_to_cm
import matplotlib


print(matplotlib.get_backend())



## Set up values

In [ ]:
mormyromast_sensor_min = AGENT_PARAMS["mormyromast_sensor_min"]  # V/m
mormyromast_sensor_max = AGENT_PARAMS["mormyromast_sensor_max"]  # V/m
morm_max_multiplier = AGENT_PARAMS["morm_max_multiplier"]
morm_min_multiplier = AGENT_PARAMS["morm_min_multiplier"]
knollen_sensor_min = AGENT_PARAMS["knollen_sensor_min"]  # V/m
knollen_sensor_max = AGENT_PARAMS["knollen_sensor_max"]  # V/m
contour_levels = np.log10([mormyromast_sensor_min, mormyromast_sensor_max])
fish_charge = AGENT_PARAMS["monopole_charges"][0]
contour_levels_morm_self_image_food_fish = np.log10([mormyromast_sensor_min * morm_min_multiplier, mormyromast_sensor_max * (1 + morm_max_multiplier)])

print("mormyromast_sensor_min", mormyromast_sensor_min)
print("fish_charge", fish_charge)
print("contour_levels_morm_self_image_food_fish", contour_levels_morm_self_image_food_fish)

## Mormyromast Self-EOD with Food

In [ ]:
importlib.reload(utils_calibrate2)
from utils_calibrate2 import (
    run_self_image_experiment,
)
criterion = ["max", "min"][0]
summary_readings, fish, all_readings, baseline = run_self_image_experiment(
    fish_orientation=np.pi / 2,
    num_points=50,
    buffer_cm=12,
    arena_size=(100, 100),
    fish_charge=fish_charge,
    do_self_induced=False,
    do_baseline_subtraction=True,
    criterion=criterion,
    no_food=False
)   
# clip to 1e-25
summary_readings[:, 2] = np.clip(
    summary_readings[:, 2], 1e-25, None
)
print("readings", summary_readings[:, 2])
baseline_max = np.max(np.abs(baseline))
# baseline_max = np.abs(baseline)[0]
# contour_levels_morm_self_image_food_fish = np.log10([baseline_max * morm_min_multiplier, baseline_max * (morm_max_multiplier)])
# baseline_vec = np.abs(baseline)
# baseline = baseline_vec[7]
contour_levels_morm_self_image_food_fish = np.log10([baseline_max * morm_min_multiplier, baseline_max * (morm_max_multiplier)])
# contour_levels_morm_self_image_food_fish = np.log10([morm_min_multiplier * (baseline * morm_max_multiplier), baseline * (morm_max_multiplier)])

print("contour_levels_morm_self_image_food_fish", contour_levels_morm_self_image_food_fish)
plot_image_fish2(
    readings=summary_readings,
    fish1=None,
    fish2=fish,
    type="self",
    contour_levels=contour_levels_morm_self_image_food_fish,  # morm min and max
    criterion=criterion,
)


## Mormyromast Self-EOD with Other Fish

In [ ]:
importlib.reload(utils_calibrate2)
from utils_calibrate2 import run_self_image_active_sense_other_fish_experiment


readings_self, fish = run_self_image_active_sense_other_fish_experiment(
    fish_orientation=np.pi / 2,
    other_fish_orientation=np.pi / 2,
    num_points=50,
    buffer=12,  # cm buffer around fish body
    fish_charge=fish_charge,  # C
    do_self_induced=False,
    do_baseline_subtraction=True,
    criterion="max",
)

print("contour_levels_morm_self_image_food_fish", contour_levels_morm_self_image_food_fish)
plot_image_fish2(
    readings=readings_self,
    fish1=None,
    fish2=fish,
    type="self",
    contour_levels=contour_levels_morm_self_image_food_fish,
    criterion="max",
)

## Ampullary Sensing Food

In [ ]:
importlib.reload(utils_calibrate2)
importlib.reload(cfg)
from utils_calibrate2 import run_sense_food_ampullary


max_readings_self, fish = run_sense_food_ampullary(
    fish_orientation=np.pi / 2,
    num_points=50,
)

ampullary_sensor_min = AGENT_PARAMS["ampullary_sensor_min"]  # V/m
contour_levels_ampullary_self_image = np.log10(
    [ampullary_sensor_min, AGENT_PARAMS["ampullary_sensor_max"]]
)  # V/m
print("ampullary_sensor_min", ampullary_sensor_min)
print("fish_charge", fish_charge)


plot_image_fish2(
    readings=max_readings_self,
    fish1=None,
    fish2=fish,
    type="self",
    contour_levels=contour_levels_ampullary_self_image,
)

## Ampullary Sensing Other Fish

In [ ]:
importlib.reload(utils_calibrate2)
importlib.reload(cfg)
from utils_calibrate2 import run_sense_other_fish_ampullary


max_readings_self, fish = run_sense_other_fish_ampullary(
    fish_orientation=np.pi / 2,
    other_fish_orientation=np.pi / 2,
    num_points=50,
)

plot_image_fish2(
    readings=max_readings_self,
    fish1=None,
    fish2=fish,
    type="self",
    contour_levels=contour_levels_ampullary_self_image,
)

## Knollen Sensing Cons-EOD 
### Directly/passively sensing cons-EOD (not "exactly" cons-image)
* Move Fish 1 around and sense on Fish 2 

In [ ]:
importlib.reload(utils_calibrate2)
importlib.reload(electric)
import arena
from utils_calibrate2 import (
    run_direct_eod_sensing_experiment_no_food,
)

knollen_sensor_min = AGENT_PARAMS["knollen_sensor_min"]  # V/m
print("knollen_sensor_min", knollen_sensor_min)
print("fish_charge", fish_charge)

contour_levels_knoll_sense_cons_eod = np.log10([knollen_sensor_min])  # V/m
print(f"Contour levels: {contour_levels_knoll_sense_cons_eod}")
max_readings_direct, fish2 = run_direct_eod_sensing_experiment_no_food(
    num_points=50,
    do_self_induced=False,
    do_induced=True,
    arena_size=(300, 300),
    buffer_cm=150,
    fish_charge=fish_charge,
    criterion='max'
)

plot_image_fish2(
    readings=max_readings_direct,
    fish1=None,
    fish2=fish2,
    type="cons",
    contour_levels=contour_levels_knoll_sense_cons_eod,

)

## Ampullary Sensing Cons-EOD 
### Ampullary is saturated by cons-EODs

In [ ]:
importlib.reload(utils_calibrate2)
importlib.reload(electric)
import arena
from utils_calibrate2 import (
    run_direct_eod_sensing_experiment_no_food_ampullary,
)

ampullary_sensor_min = AGENT_PARAMS["ampullary_sensor_min"]  # V/m
ampullary_sensor_max = AGENT_PARAMS["ampullary_sensor_max"]  # V/m
contour_levels_amp_sense_cons_eod = np.log10([ampullary_sensor_min, ampullary_sensor_max])  # V/m
print(f"Contour levels: {contour_levels_amp_sense_cons_eod}")
max_readings_direct, fish2 = run_direct_eod_sensing_experiment_no_food_ampullary(
    num_points=50,
    arena_size=(100000, 100000),
    buffer_cm=50000,
)

plot_image_fish2(
    readings=max_readings_direct,
    fish1=None,
    fish2=fish2,
    type="cons",
    contour_levels=contour_levels_amp_sense_cons_eod,
)

## TODO Add in visualization from our multi-baseline/virtual sensor model
## Directly/passively sensing cons-EOD + induced food dipole
* Move a food object around in the presence of Fish 1 emitting and Fish 2 receiving
* Explore a set of Fish 1 (emitter) vs Fish 2 (receiver) configurations across different runs
* Configurations: 
  (1) Fish 1 (facing left) is to the upper right of Fish 2 (facing up), with distance d between them
  (2) Fish 1 (facing up) is above Fish 2 (facing up), with distance d between them
  (3) Fish 1 (facing up) is below Fish 2 (facing up), with distance d between them


In [ ]:
# import importlib
# import utils_calibrate2  # assuming the functions are in this file

# importlib.reload(utils_calibrate2)
# from utils_calibrate2 import (
#     run_direct_eod_sensing_experiment_with_food,
#     plot_image_fish2,
# )

# for fish_config in [2, 1, 3]:
#     print(f"--- Running Experiment for Configuration {fish_config} ---")
#     max_readings_direct, fish1, fish2 = run_direct_eod_sensing_experiment_with_food(
#         num_points=50,
#         do_self_induced=[True, False][1],
#         arena_size=(100, 100),
#         buffer_cm=25,
#         dist_fish=10,
#         fish_config=fish_config,
#         dynamic_baseline=[None, "no_food", "mean", "min_max", "spatial"][0],
#         fish_charge=fish_charge,
#         criterion='min',
#     )
#     max_readings_direct = np.array(max_readings_direct)
#     print(np.log10(max_readings_direct[:,2]))
#     # max_readings_direct[:, 2] *= 100  # Scale the readings for better visualization

#     plot_image_fish2(
#         readings=max_readings_direct,
#         fish1=fish1,
#         fish2=fish2,
#         type="cons",
#         # contour_levels=np.log10([0.001, 0.01]),
#         # contour_levels=contour_levels_morm_self_image_food_fish,
#         contour_levels=contour_levels_morm_no_self_EOD,
#         log_scale=True,
#         criterion='min',
#     )

In [ ]:
# # Can bring into standard mormyromast range by scaling the readings
# max_readings_direct_scaled = max_readings_direct.copy()
# max_readings_direct_scaled[:, 2] = (
#     max_readings_direct_scaled[:, 2] * 100
# )  # Scale the readings
# print(contour_levels_morm_self_image_food_fish)
# plot_image_fish2(
#     max_readings=max_readings_direct_scaled,
#     fish1=fish1,
#     fish2=fish2,
#     type="cons",
#     # contour_levels=np.log10([0.001, 0.01]),
#     contour_levels=contour_levels_morm_self_image_food_fish,
# )

In [ ]:
# import numpy as np
# import importlib
# import utils_calibrate2  # assuming the functions are in this file

# importlib.reload(utils_calibrate2)
# from utils_calibrate2 import (
#     run_direct_eod_sensing_experiment_fixed_midpoint_food_vary_fish,
#     plot_image_fish2,
# )

# readings, fish1_init, fish2 = run_direct_eod_sensing_experiment_fixed_midpoint_food_vary_fish(
#     num_points=50,
#     do_self_induced=False,
#     arena_size=(100, 100),
#     buffer_cm=15,
#     dist_fish=10,
#     fish_config=2
# )

# plot_image_fish2(
#     readings=readings,
#     fish1=fish1_init,
#     fish2=fish2,
#     type="cons",
#     contour_levels=contour_levels_morm_self_image_food_fish,
# )

In [ ]:
# Deprecated but might possibly offer a good template with the passed-in baseline_val for visualizing the virtual sensors

# importlib.reload(utils_calibrate2)
# from utils_calibrate2 import (
#     virtual_sense_food,
# )
# criterion = ["max", "min"][0]
# # baseline_vals = AGENT_PARAMS["mormyromast_baselines"].flatten()
# baseline_vals = [1]
# new_contour_levels_morm_self_image_food_fish = np.log10([4e-7, 1e-1])  # V/m
# # new_contour_levels_morm_self_image_food_fish = np.log10([3.98e-07, 8.87e-02])  # V/m

# for baseline_val in baseline_vals:

#     summary_readings, fish, all_readings = virtual_sense_food(
#         fish_orientation=np.pi / 2,
#         num_points=50,
#         buffer_cm=12,
#         arena_size=(100, 100),
#         fish_charge=fish_charge,
#         do_self_induced=False,
#         criterion=criterion,
#         no_food=False,
#         virtual_baseline_multiplier=baseline_val,
#         sensor_idx=None
#     )

#     plot_image_fish2(
#         readings=summary_readings,
#         fish1=None,
#         fish2=fish,
#         type="self",
#         contour_levels=new_contour_levels_morm_self_image_food_fish,  # morm min and max
#         criterion=criterion,
#     )



In [ ]:
# Deprecated since changing to randomized virtual sensors

# importlib.reload(utils_calibrate2)
# from utils_calibrate2 import (
#     virtual_sense_agent_food,
#     plot_image_fish2,
#     plot_line_fish2,
# )
# sample_mode = ["grid", "line"][0]
# criterion = ["max", "min"][0]
# # baseline_vals = AGENT_PARAMS["mormyromast_baselines"].flatten()
# baseline_vals = [1, 1, 1, 1, 1]
# baseline_dists = [0, 3, 5, 7, 9]
# new_contour_levels_morm_self_image_food_fish = np.log10([4e-7, 1e-3]) # V/m
# dist_fish = [3, 5, 7, 9][3]
# fish_config = [1, 2, 3][1]

# print("baseline_vals", baseline_vals)


# for baseline_val, baseline_dist in zip(baseline_vals, baseline_dists):
#     if baseline_dist == 0: continue
#     # if baseline_dist != 9: continue

#     summary_readings, fish1, fish2 = virtual_sense_agent_food(
#         num_points=50 if sample_mode == "grid" else 100,
#         buffer_cm=12,
#         dist_fish=baseline_dist,
#         fish_config=fish_config,
#         arena_size=(100, 100),
#         fish_charge=fish_charge,
#         do_self_induced=True,
#         criterion=criterion,
#         virtual_baseline_multiplier=baseline_val,
#         sensor_idx=18,
#         sample_mode=sample_mode,
#         fish_food_buffer_cm=1.25, # needed for line-sampling sensitivity
#         no_food=False,
#     )

#     no_food_readings, fish1, fish2 = virtual_sense_agent_food(
#         num_points=50 if sample_mode == "grid" else 100,
#         buffer_cm=12,
#         dist_fish=baseline_dist,
#         fish_config=fish_config,
#         arena_size=(100, 100),
#         fish_charge=fish_charge,
#         do_self_induced=True,
#         criterion=criterion,
#         virtual_baseline_multiplier=baseline_val,
#         sensor_idx=18,
#         sample_mode=sample_mode,
#         fish_food_buffer_cm=1.25, # needed for line-sampling sensitivity
#         no_food=True,
#     )

#     # summary_readings[:, 2] = np.abs(summary_readings[:, 2])
#     if sample_mode == "grid":
#         plot_image_fish2(
#             readings=summary_readings,
#             fish1=fish1,
#             fish2=fish2,
#             type="self" if baseline_dist == 0 else "cons",
#             contour_levels=new_contour_levels_morm_self_image_food_fish,  # morm min and max
#             criterion="idx",
#         )
#     if sample_mode == "line":
#         plot_line_fish2(
#             summary_readings,
#             no_food_readings,
#             fish1=fish1,
#             fish2=fish2,
#             type="cons",
#             log_scale=True,
#             criterion="max",
#             annotate_agents=True,
#             fish_food_buffer_cm=1.25,
#         )



In [ ]:
# for collecting data along the center line between the two fish

def get_center_line_data(
        summary_readings,
        fish1,
        fish2,
        arena_size=(100, 100),
        buffer_cm=12,
        num_points=50,
        ):# summary_readings is assumed to be (N, 3) with columns [x, y, value]
    center = np.array(arena_size) / 2.0
    # grid spacing in x:
    dx = (2*buffer_cm) / (num_points - 1)
    # pick points whose x is on the vertical center line (between the fish)
    mask = np.isclose(summary_readings[:, 0], center[0], atol=dx/2 + 1e-9)
    line_pts = summary_readings[mask]  # shape (~num_points, 3)

    if line_pts.size == 0:
        raise RuntimeError("No points found on the center line. Check num_points/buffer_cm.")

    # Optionally focus only on the *segment* between the fish (not the whole center line window).
    y1, y2 = fish2.position[1], fish1.position[1]
    y_low, y_high = (y1, y2) if y1 < y2 else (y2, y1)
    between_mask = (line_pts[:, 1] >= y_low) & (line_pts[:, 1] <= y_high)
    line_between = line_pts[between_mask] if np.any(between_mask) else line_pts

    vals = line_between[:, 2]
    ys = line_between[:, 1]
    return vals, ys



def plot_center_line_data(electric_readings, dists, no_food_readings=None, no_food_dists=None):
    plt.figure(figsize=(6, 3))
    order = np.argsort(dists)
    dists = dists[order]
    electric_readings = electric_readings[order]
    plt.plot(dists, electric_readings, marker='o')
    if no_food_readings is not None:
        order = np.argsort(no_food_dists)
        no_food_readings = no_food_readings[order]
        no_food_dists = no_food_dists[order]
        plt.plot(dists, no_food_readings, marker='o', label='No Food', linestyle='--')
        plt.legend()
    # plt.axhline(np.mean(electric_readings)*0.75, color='gray', linestyle='--', label='Mean - 25%')
    # plt.axhline(np.mean(electric_readings)*1.25, color='gray', linestyle='--', label='Mean + 25%')
    plt.yscale("log")
    plt.xlabel("Position (cm)")
    plt.ylabel("Electric Field (V/m)")
    plt.title(f"Sensed Field vs y position between fish")
    plt.tight_layout()
    plt.show()